# LeetCode 1818 — Minimum Absolute Sum Difference bằng Python

Link đề: https://leetcode.com/problems/minimum-absolute-sum-difference/ · Độ khó: **Medium** · Chủ đề: Array, Binary Search, Sorting, Ordered Set

Skeleton LeetCode:

```python
class Solution:
    def minAbsoluteSumDiff(self, nums1: List[int], nums2: List[int]) -> int:
        ...
```

Lộ trình: **thử mọi phép thay rồi tính lại tổng** `O(n³)` → **tính "lợi ích" của từng phép thay** `O(n²)` → **sắp xếp `nums1` + tìm kiếm nhị phân giá trị gần nhất** `O(n log n)` (lời giải nộp) → **sắp xếp cả hai + hai con trỏ** (quét hòa nhập).

## 1. Mục tiêu học

1. Biến bài "tối thiểu tổng sau một thao tác" thành "**tối đa lợi ích** của một thao tác": `đáp án = tổng ban đầu − lợi ích lớn nhất`.
2. Với mỗi vị trí `i`, tìm phần tử của `nums1` **gần `nums2[i]` nhất** bằng `bisect` trên mảng đã sắp xếp — và nhớ xét **cả hai** láng giềng.
3. Hiểu vì sao chiến lược tham lam "sửa vị trí chênh lệch lớn nhất" là **sai**.
4. Áp dụng **modulo** đúng: chỉ lấy dư ở **kết quả cuối**, không so sánh các giá trị đã lấy dư.
5. Kiểm chứng bằng vét cạn ngẫu nhiên có seed.

## 2. Đề bài & ví dụ

Cho hai mảng số nguyên dương `nums1`, `nums2`, cùng độ dài `n`.

**Tổng chênh lệch tuyệt đối** của hai mảng là `Σ |nums1[i] − nums2[i]|` với `0 ≤ i < n`.

Bạn được **thay tối đa một** phần tử của `nums1` bằng **một phần tử khác của `nums1`** (giá trị lấy từ chính `nums1`) để tổng chênh lệch nhỏ nhất. Trả về tổng nhỏ nhất đó, **modulo `10⁹ + 7`**.

**Ví dụ 1:** `nums1 = [1,7,5], nums2 = [2,3,5]` → `3`. Thay `7` bằng `1` hoặc `5`: `|1−2| + (|1−3| hoặc |5−3|) + |5−5| = 3`.

**Ví dụ 2:** `nums1 = [2,4,6,8,10], nums2 = [2,4,6,8,10]` → `0`.

**Ví dụ 3:** `nums1 = [1,10,4,4,2,7], nums2 = [9,3,5,1,7,4]` → `20`. Thay `nums1[0] = 1` bằng `10`: `|10−9| + |10−3| + |4−5| + |4−1| + |2−7| + |7−4| = 20`.

**Ràng buộc:** `1 ≤ n ≤ 10⁵`, `1 ≤ nums1[i], nums2[i] ≤ 10⁵`.

In [1]:
import random
import time
from bisect import bisect_left
from typing import List

MOD = 10**9 + 7
examples = [
    ([1, 7, 5], [2, 3, 5], 3),
    ([2, 4, 6, 8, 10], [2, 4, 6, 8, 10], 0),
    ([1, 10, 4, 4, 2, 7], [9, 3, 5, 1, 7, 4], 20),
]
for a, b, want in examples:
    print(a, b, "-> kỳ vọng", want)

[1, 7, 5] [2, 3, 5] -> kỳ vọng 3
[2, 4, 6, 8, 10] [2, 4, 6, 8, 10] -> kỳ vọng 0
[1, 10, 4, 4, 2, 7] [9, 3, 5, 1, 7, 4] -> kỳ vọng 20


## 3. Kiến thức nền

### 3.1 `zip` và tổng chênh lệch

In [2]:
a, b = [1, 7, 5], [2, 3, 5]
print([abs(x - y) for x, y in zip(a, b)], "-> tổng", sum(abs(x - y) for x, y in zip(a, b)))

[1, 4, 0] -> tổng 5


### 3.2 `bisect_left` trên mảng đã sắp xếp

`bisect_left(arr, x)` trả vị trí `p` nhỏ nhất mà `arr[p] ≥ x` (hoặc `len(arr)` nếu mọi phần tử `< x`). Phần tử **gần `x` nhất** là `arr[p]` (nhỏ nhất `≥ x`) hoặc `arr[p−1]` (lớn nhất `< x`) — phải xét **cả hai**, và kiểm biên.

In [3]:
arr = sorted([1, 10, 4, 4, 2, 7])
print("arr =", arr)
for x in [0, 3, 5, 9, 11]:
    p = bisect_left(arr, x)
    cands = [arr[q] for q in (p - 1, p) if 0 <= q < len(arr)]
    print(f"x={x:>2}: p={p}, láng giềng {cands} -> gần nhất {min(cands, key=lambda v: abs(v - x))}")

arr = [1, 2, 4, 4, 7, 10]
x= 0: p=0, láng giềng [1] -> gần nhất 1
x= 3: p=2, láng giềng [2, 4] -> gần nhất 2
x= 5: p=4, láng giềng [4, 7] -> gần nhất 4
x= 9: p=5, láng giềng [7, 10] -> gần nhất 10
x=11: p=6, láng giềng [10] -> gần nhất 10


### 3.3 Đổi "tối thiểu tổng" thành "tối đa lợi ích"

Thay `nums1[i]` bằng `v` chỉ làm thay đổi **một** số hạng: từ `|nums1[i] − nums2[i]|` thành `|v − nums2[i]|`. Lợi ích `gain_i = |nums1[i] − nums2[i]| − min_v |v − nums2[i]|` (≥ 0 vì có thể chọn `v = nums1[i]`). Đáp án = `tổng − max_i gain_i`.

In [4]:
a, b = [1, 7, 5], [2, 3, 5]
for i in range(3):
    d1 = abs(a[i] - b[i]); d2 = min(abs(v - b[i]) for v in a)
    print(f"i={i}: hiện tại {d1}, tốt nhất {d2}, lợi ích {d1 - d2}")

i=0: hiện tại 1, tốt nhất 1, lợi ích 0
i=1: hiện tại 4, tốt nhất 2, lợi ích 2
i=2: hiện tại 0, tốt nhất 0, lợi ích 0


### 3.4 Modulo và phép so sánh

`x % MOD` **không giữ thứ tự**: `MOD + 5 > MOD − 1` nhưng `(MOD + 5) % MOD = 5 < MOD − 1`. Vì vậy phải tìm giá trị nhỏ nhất trên **số thật**, rồi mới lấy dư.

In [5]:
x, y = MOD + 5, MOD - 1
print(x > y, x % MOD > y % MOD)

True False


### 3.5 Hai con trỏ trên hai dãy đã sắp xếp

Nếu cả truy vấn (`nums2[i]`) cũng được xử lý theo thứ tự tăng, con trỏ vào mảng `nums1` đã sắp xếp chỉ cần **tiến**, không cần tìm kiếm nhị phân lại.

In [6]:
arr = [1, 2, 4, 4, 7, 10]
queries = sorted([9, 3, 5, 1, 7, 4])
p = 0
for x in queries:
    while p < len(arr) and arr[p] < x:
        p += 1
    print(f"x={x}: con trỏ dừng ở p={p} (arr[p] = {arr[p] if p < len(arr) else None})")

x=1: con trỏ dừng ở p=0 (arr[p] = 1)
x=3: con trỏ dừng ở p=2 (arr[p] = 4)
x=4: con trỏ dừng ở p=2 (arr[p] = 4)
x=5: con trỏ dừng ở p=4 (arr[p] = 7)
x=7: con trỏ dừng ở p=4 (arr[p] = 7)
x=9: con trỏ dừng ở p=5 (arr[p] = 10)


## 4. Ý tưởng nền tảng / trực giác

Ví dụ 3 `nums1 = [1,10,4,4,2,7]`, `nums2 = [9,3,5,1,7,4]`. Tổng ban đầu: `8 + 7 + 1 + 3 + 5 + 3 = 27`.

Với từng vị trí, phần tử của `nums1` gần `nums2[i]` nhất và lợi ích:

| i | nums1[i] | nums2[i] | hiện tại | gần nhất trong nums1 | tốt nhất | lợi ích |
|---|---|---|---|---|---|---|
| 0 | 1 | 9 | 8 | 10 | 1 | **7** |
| 1 | 10 | 3 | 7 | 2 hoặc 4 | 1 | 6 |
| 2 | 4 | 5 | 1 | 4 | 1 | 0 |
| 3 | 4 | 1 | 3 | 1 | 0 | 3 |
| 4 | 2 | 7 | 5 | 7 | 0 | 5 |
| 5 | 7 | 4 | 3 | 4 | 0 | 3 |

Lợi ích lớn nhất 7 (thay `nums1[0]` bằng 10) ⇒ `27 − 7 = 20`. ✔

Chú ý: vị trí chênh lệch lớn nhất (`i = 0`, chênh 8) tình cờ cũng là vị trí lợi ích lớn nhất ở đây — nhưng **không phải luôn vậy** (Lỗi 2).

## 5. Cách 1 — Brute Force / baseline: thử mọi phép thay, tính lại tổng

In [7]:
def min_sum_brute(nums1, nums2):
    n = len(nums1)
    best = sum(abs(x - y) for x, y in zip(nums1, nums2))        # không thay
    for i in range(n):
        for j in range(n):
            arr = nums1[:]; arr[i] = nums1[j]
            best = min(best, sum(abs(x - y) for x, y in zip(arr, nums2)))
    return best % MOD

for a, b, want in examples:
    print(min_sum_brute(a, b), "| kỳ vọng", want)

3 | kỳ vọng 3
0 | kỳ vọng 0
20 | kỳ vọng 20


**Độ phức tạp:** `n²` phép thay × `O(n)` tính tổng ⇒ `O(n³)`. Nút thắt: phép thay chỉ đổi **một** số hạng, không cần tính lại cả tổng.

## 6. Ý tưởng thuật toán / bước đột phá

**Insight 1:** đáp án = `S − max_i gain_i`, `gain_i = |a_i − b_i| − min_{v ∈ nums1} |v − b_i|`.

**Insight 2:** `min_{v} |v − b_i|` = khoảng cách từ `b_i` tới phần tử gần nhất trong **tập** giá trị của `nums1` ⇒ sắp xếp `nums1` một lần, mỗi truy vấn tìm nhị phân `O(log n)`.

**Quy tắc "nếu… thì…":**
- `p = bisect_left(sorted1, b_i)`.
- *Nếu* `p < n` *thì* ứng viên `sorted1[p]` (nhỏ nhất `≥ b_i`).
- *Nếu* `p > 0` *thì* ứng viên `sorted1[p − 1]` (lớn nhất `< b_i`).
- `best_i` = min khoảng cách hai ứng viên; `gain_i = |a_i − b_i| − best_i`.

**Lưu ý modulo:** tính `S` và `max gain` trên số nguyên thật (Python không tràn), rồi `(S − max_gain) % MOD`. `S` có thể tới `10⁵ · 10⁵ = 10¹⁰ > MOD`.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Lợi ích từng vị trí, tìm gần nhất bằng quét (`O(n²)`)

In [8]:
def min_sum_gain_quadratic(nums1, nums2):
    S = sum(abs(x - y) for x, y in zip(nums1, nums2))
    best_gain = 0
    for a, b in zip(nums1, nums2):
        d2 = min(abs(v - b) for v in nums1)              # quét O(n)
        best_gain = max(best_gain, abs(a - b) - d2)
    return (S - best_gain) % MOD

for a, b, want in examples:
    print(min_sum_gain_quadratic(a, b), want)

3 3
0 0
20 20


**Độ phức tạp:** `O(n²)` — `10¹⁰` với `n = 10⁵`. Nút thắt: tìm "phần tử gần nhất" bằng quét; dùng **sắp xếp + nhị phân**.

### 7.2 Cách 3 — Sắp xếp `nums1` + `bisect` (`O(n log n)`) — lời giải chuẩn

In [9]:
def min_sum_bisect_demo(nums1, nums2):
    srt = sorted(nums1)
    S = sum(abs(x - y) for x, y in zip(nums1, nums2))
    best_gain = 0
    for i, (a, b) in enumerate(zip(nums1, nums2)):
        p = bisect_left(srt, b)
        d2 = min(abs(srt[q] - b) for q in (p - 1, p) if 0 <= q < len(srt))
        g = abs(a - b) - d2
        best_gain = max(best_gain, g)
        print(f"i={i}: b={b}, p={p}, gần nhất cách {d2}, lợi ích {g}, max={best_gain}")
    print("S =", S)
    return (S - best_gain) % MOD

print("=>", min_sum_bisect_demo(*examples[2][:2]))

i=0: b=9, p=5, gần nhất cách 1, lợi ích 7, max=7
i=1: b=3, p=2, gần nhất cách 1, lợi ích 6, max=7
i=2: b=5, p=4, gần nhất cách 1, lợi ích 0, max=7
i=3: b=1, p=0, gần nhất cách 0, lợi ích 3, max=7
i=4: b=7, p=4, gần nhất cách 0, lợi ích 5, max=7
i=5: b=4, p=2, gần nhất cách 0, lợi ích 3, max=7
S = 27
=> 20


In [10]:
class SolutionBisect:
    def minAbsoluteSumDiff(self, nums1: List[int], nums2: List[int]) -> int:
        srt = sorted(nums1)
        n = len(srt)
        S = 0
        best_gain = 0
        for a, b in zip(nums1, nums2):
            d1 = abs(a - b)
            S += d1
            p = bisect_left(srt, b)
            d2 = d1
            if p < n:
                d2 = min(d2, srt[p] - b)
            if p > 0:
                d2 = min(d2, b - srt[p - 1])
            best_gain = max(best_gain, d1 - d2)
        return (S - best_gain) % MOD

for a, b, want in examples:
    print(SolutionBisect().minAbsoluteSumDiff(a, b) == want)

True
True
True


**Độ phức tạp:** sắp xếp `O(n log n)` + `n` lần `bisect` `O(log n)` ⇒ `O(n log n)`; bộ nhớ `O(n)`.

### 7.3 Cách 4 — Sắp xếp cả hai, hai con trỏ (quét hòa nhập)

Xử lý các vị trí theo thứ tự `nums2[i]` tăng dần; con trỏ `p` vào `sorted(nums1)` chỉ tiến. Bỏ được `log n` cho phần truy vấn (phần sắp xếp vẫn `O(n log n)`), và là kỹ thuật hay khi "nhiều truy vấn gần nhất trên một tập cố định".

In [11]:
class SolutionMerge:
    def minAbsoluteSumDiff(self, nums1: List[int], nums2: List[int]) -> int:
        srt = sorted(nums1)
        n = len(srt)
        order = sorted(range(n), key=lambda i: nums2[i])
        S = sum(abs(x - y) for x, y in zip(nums1, nums2))
        best_gain, p = 0, 0
        for i in order:
            b = nums2[i]
            while p < n and srt[p] < b:
                p += 1
            d2 = float("inf")
            if p < n: d2 = srt[p] - b
            if p > 0: d2 = min(d2, b - srt[p - 1])
            best_gain = max(best_gain, abs(nums1[i] - b) - d2)
        return (S - best_gain) % MOD

for a, b, want in examples:
    print(SolutionMerge().minAbsoluteSumDiff(a, b) == want)

True
True
True


**Độ phức tạp:** `O(n log n)` (hai lần sắp xếp) + `O(n)` quét; bộ nhớ `O(n)`.

## 8. TRACE từng bước

Trace Cách 3 trên Ví dụ 1: `nums1 = [1,7,5]`, `nums2 = [2,3,5]`, `sorted(nums1) = [1,5,7]`.

| i | a | b | `d1 = |a−b|` | `p = bisect_left` | ứng viên | `d2` | lợi ích | `S` | max lợi ích |
|---|---|---|---|---|---|---|---|---|---|
| 0 | 1 | 2 | 1 | 1 | 1, 5 | 1 | 0 | 1 | 0 |
| 1 | 7 | 3 | 4 | 1 | 1, 5 | 2 | **2** | 5 | 2 |
| 2 | 5 | 5 | 0 | 1 | 1, 5 | 0 | 0 | 5 | 2 |

Đáp án `5 − 2 = 3`. ✔

In [12]:
def trace_bisect(nums1, nums2):
    srt = sorted(nums1); n = len(srt)
    print("sorted(nums1) =", srt)
    S = best = 0
    for i, (a, b) in enumerate(zip(nums1, nums2)):
        d1 = abs(a - b); S += d1
        p = bisect_left(srt, b)
        left = srt[p - 1] if p > 0 else None
        right = srt[p] if p < n else None
        d2 = min([d1] + [abs(v - b) for v in (left, right) if v is not None])
        best = max(best, d1 - d2)
        print(f"i={i} a={a} b={b} d1={d1} p={p} trái={left} phải={right} d2={d2} lợi ích={d1-d2} S={S} max={best}")
    print("đáp án =", (S - best) % MOD)
    return (S - best) % MOD

trace_bisect([1, 7, 5], [2, 3, 5])

sorted(nums1) = [1, 5, 7]
i=0 a=1 b=2 d1=1 p=1 trái=1 phải=5 d2=1 lợi ích=0 S=1 max=0
i=1 a=7 b=3 d1=4 p=1 trái=1 phải=5 d2=2 lợi ích=2 S=5 max=2
i=2 a=5 b=5 d1=0 p=1 trái=1 phải=5 d2=0 lợi ích=0 S=5 max=2
đáp án = 3


3

In [13]:
trace_bisect([1, 10, 4, 4, 2, 7], [9, 3, 5, 1, 7, 4])

sorted(nums1) = [1, 2, 4, 4, 7, 10]
i=0 a=1 b=9 d1=8 p=5 trái=7 phải=10 d2=1 lợi ích=7 S=8 max=7
i=1 a=10 b=3 d1=7 p=2 trái=2 phải=4 d2=1 lợi ích=6 S=15 max=7
i=2 a=4 b=5 d1=1 p=4 trái=4 phải=7 d2=1 lợi ích=0 S=16 max=7
i=3 a=4 b=1 d1=3 p=0 trái=None phải=1 d2=0 lợi ích=3 S=19 max=7
i=4 a=2 b=7 d1=5 p=4 trái=4 phải=7 d2=0 lợi ích=5 S=24 max=7
i=5 a=7 b=4 d1=3 p=2 trái=2 phải=4 d2=0 lợi ích=3 S=27 max=7
đáp án = 20


20

## 9. Code nộp LeetCode

In [14]:
from bisect import bisect_left
from typing import List

class Solution:
    def minAbsoluteSumDiff(self, nums1: List[int], nums2: List[int]) -> int:
        MOD = 10**9 + 7
        srt = sorted(nums1)
        n = len(srt)
        total = 0
        best_gain = 0
        for a, b in zip(nums1, nums2):
            d1 = abs(a - b)
            total += d1
            p = bisect_left(srt, b)          # srt[p] >= b > srt[p-1]
            d2 = d1
            if p < n:
                d2 = min(d2, srt[p] - b)
            if p > 0:
                d2 = min(d2, b - srt[p - 1])
            best_gain = max(best_gain, d1 - d2)
        return (total - best_gain) % MOD     # lấy dư SAU khi trừ, trên số thật

for a, b, want in examples:
    assert Solution().minAbsoluteSumDiff(a, b) == want
print("Ví dụ đề: OK")

Ví dụ đề: OK


## 10. Vì sao thuật toán đúng

**Bổ đề 1:** gọi `S = Σ|a_i − b_i|`. Phép thay `a_i ← v` (với `v ∈ nums1`) cho tổng mới `S − |a_i − b_i| + |v − b_i|`. Vậy tổng nhỏ nhất sau tối đa một phép thay là `S − max(0, max_i (|a_i − b_i| − min_v |v − b_i|)) = S − max_i gain_i` (mỗi `gain_i ≥ 0` vì chọn `v = a_i` được). Lưu ý giá trị `v` lấy từ `nums1` **ban đầu** — đúng như đề (thay bằng một phần tử khác của `nums1`).

**Bổ đề 2:** trong mảng tăng `srt`, phần tử gần `b` nhất là `srt[p]` hoặc `srt[p − 1]` với `p = bisect_left(srt, b)`. *Chứng minh:* mọi `srt[q]` với `q > p` có `srt[q] ≥ srt[p] ≥ b` nên xa hơn hoặc bằng `srt[p]`; mọi `srt[q]` với `q < p − 1` có `srt[q] ≤ srt[p−1] < b` nên xa hơn hoặc bằng `srt[p−1]`.

**Modulo:** `S` và `best_gain` được tính trên số nguyên chính xác (Python), phép so sánh `max` thực hiện trên giá trị thật; chỉ kết quả cuối được lấy dư ⇒ đúng.

**Cách 4:** vì xử lý `b` tăng dần, `p` (vị trí đầu tiên `srt[p] ≥ b`) không giảm ⇒ con trỏ chỉ tiến, cho cùng `p` như `bisect_left`.

## 11. Phân tích Time / Space

| Cách | Time | Space |
|---|---|---|
| 1. Thử mọi phép thay | `O(n³)` | `O(n)` |
| 2. Lợi ích + quét | `O(n²)` | `O(1)` phụ |
| 3. Sắp xếp + bisect | `O(n log n)` | `O(n)` |
| 4. Hai con trỏ | `O(n log n)` (sắp xếp) + `O(n)` | `O(n)` |

In [15]:
rng = random.Random(1818)
n = 10**5
A = [rng.randint(1, 10**5) for _ in range(n)]
B = [rng.randint(1, 10**5) for _ in range(n)]
for name, sol in [("Cách 3", SolutionBisect()), ("Cách 4", SolutionMerge()), ("Nộp", Solution())]:
    t = time.perf_counter(); r = sol.minAbsoluteSumDiff(A, B)
    print(f"{name}: {r} trong {1000*(time.perf_counter()-t):.0f} ms")
t = time.perf_counter(); min_sum_gain_quadratic(A[:1500], B[:1500]); dt = time.perf_counter() - t
print(f"Cách 2 với n=1500: {dt:.2f}s -> n=10^5 ước tính ≈ {dt * (n / 1500) ** 2:.0f}s")

Cách 3: 333378206 trong 87 ms


Cách 4: 333378206 trong 105 ms
Nộp: 333378206 trong 81 ms


Cách 2 với n=1500: 0.09s -> n=10^5 ước tính ≈ 406s


## 12. Bộ test

### 12.1 Ví dụ đề + edge case (tự dựng, đối chiếu vét cạn)

In [16]:
sols = [Solution(), SolutionBisect(), SolutionMerge()]
edge = examples + [
    ([5], [3], 2),                        # n = 1: không có phần tử khác để thay
    ([1, 1], [9, 9], 16),                 # mọi phần tử như nhau: không cải thiện được
    ([8, 9], [4, 6], 6),                  # phản ví dụ cho tham lam (Lỗi 2)
    ([1, 100000], [100000, 1], 99999),
    ([3, 3, 3], [3, 3, 3], 0),
]
for a, b, want in edge:
    assert min_sum_brute(a, b) == want
    for s in sols:
        assert s.minAbsoluteSumDiff(a, b) == want, (a, b, s)
    assert min_sum_gain_quadratic(a, b) == want
    print("OK", a, b, "->", want)

OK [1, 7, 5] [2, 3, 5] -> 3
OK [2, 4, 6, 8, 10] [2, 4, 6, 8, 10] -> 0
OK [1, 10, 4, 4, 2, 7] [9, 3, 5, 1, 7, 4] -> 20
OK [5] [3] -> 2
OK [1, 1] [9, 9] -> 16
OK [8, 9] [4, 6] -> 6
OK [1, 100000] [100000, 1] -> 99999
OK [3, 3, 3] [3, 3, 3] -> 0


### 12.2 Ngẫu nhiên có seed: so với vét cạn

In [17]:
rng = random.Random(42)
for _ in range(1500):
    n = rng.randint(1, 9)
    hi = rng.choice([5, 20, 10**5])
    a = [rng.randint(1, hi) for _ in range(n)]
    b = [rng.randint(1, hi) for _ in range(n)]
    want = min_sum_brute(a, b)
    for s in sols:
        assert s.minAbsoluteSumDiff(a, b) == want, (a, b)
for _ in range(30):
    n = rng.randint(100, 400)
    a = [rng.randint(1, 10**5) for _ in range(n)]; b = [rng.randint(1, 10**5) for _ in range(n)]
    want = min_sum_gain_quadratic(a, b)
    assert all(s.minAbsoluteSumDiff(a, b) == want for s in sols)
print("1500 case nhỏ (vét cạn) + 30 case vừa (Cách 2): khớp")

1500 case nhỏ (vét cạn) + 30 case vừa (Cách 2): khớp


### 12.3 Test vượt modulo (tự dựng)

`n = 10001`, `nums1 = [1]*(n−1) + [100000]`, `nums2 = [100000]*(n−1) + [1]`: tổng ban đầu `n · 99999 = 1000089999 > 10⁹ + 7`, mỗi vị trí có lợi ích `99999` ⇒ đáp án thật `(n − 1) · 99999 = 999990000`.

In [18]:
n_big = 10001
big1 = [1] * (n_big - 1) + [100000]
big2 = [100000] * (n_big - 1) + [1]
S_big = sum(abs(x - y) for x, y in zip(big1, big2))
print("tổng ban đầu:", S_big, "> MOD:", S_big > MOD)
for s in sols:
    assert s.minAbsoluteSumDiff(big1, big2) == 999990000
print("đáp án:", Solution().minAbsoluteSumDiff(big1, big2))

tổng ban đầu: 1000089999 > MOD: True
đáp án: 999990000


## 13. Lỗi người mới hay gặp

### Lỗi 1 — Chỉ xét một láng giềng sau `bisect_left`

In [19]:
def wrong_one_neighbor(nums1, nums2):
    srt = sorted(nums1); n = len(srt)
    S = sum(abs(x - y) for x, y in zip(nums1, nums2)); best = 0
    for a, b in zip(nums1, nums2):
        p = min(bisect_left(srt, b), n - 1)
        d2 = abs(srt[p] - b)                       # SAI: bỏ qua srt[p-1]
        best = max(best, abs(a - b) - d2)
    return (S - best) % MOD

a, b = [1, 7, 5], [2, 3, 5]
print("sai:", wrong_one_neighbor(a, b), "| đúng:", Solution().minAbsoluteSumDiff(a, b))

sai: 3 | đúng: 3


**Vì sao sai:** với `b = 2`, `bisect_left([1,5,7], 2) = 1` ⇒ chỉ xét `5` (cách 3), bỏ qua `1` (cách 1) — "lợi ích" ở `i = 0` bị tính thành `1 − 3 = −2`. Trên Ví dụ 1 kết quả **tình cờ vẫn đúng** (lợi ích lớn nhất nằm ở vị trí khác), và trên Ví dụ 3 cũng vậy. Tìm phản ví dụ bằng test ngẫu nhiên:

In [20]:
a, b = [1, 10, 4, 4, 2, 7], [9, 3, 5, 1, 7, 4]
print("sai:", wrong_one_neighbor(a, b), "| đúng:", Solution().minAbsoluteSumDiff(a, b))
rng = random.Random(1)
for _ in range(2000):
    a = [rng.randint(1, 9) for _ in range(4)]; b = [rng.randint(1, 9) for _ in range(4)]
    if wrong_one_neighbor(a, b) != Solution().minAbsoluteSumDiff(a, b):
        print("phản ví dụ:", a, b, "sai:", wrong_one_neighbor(a, b), "| đúng:", Solution().minAbsoluteSumDiff(a, b)); break

sai: 20 | đúng: 20
phản ví dụ: [7, 1, 1, 6] [3, 3, 3, 5] sai: 8 | đúng: 7


**Sửa:** xét cả `srt[p]` (nếu `p < n`) và `srt[p − 1]` (nếu `p > 0`). Bài học: một lỗi có thể **không lộ** trên ví dụ của đề — luôn test ngẫu nhiên.

### Lỗi 2 — Tham lam: sửa vị trí có chênh lệch lớn nhất

In [21]:
def wrong_greedy_max_diff(nums1, nums2):
    S = sum(abs(x - y) for x, y in zip(nums1, nums2))
    i = max(range(len(nums1)), key=lambda k: abs(nums1[k] - nums2[k]))
    d2 = min(abs(v - nums2[i]) for v in nums1)
    return (S - (abs(nums1[i] - nums2[i]) - d2)) % MOD

a, b = [8, 9], [4, 6]
print("tham lam:", wrong_greedy_max_diff(a, b), "| đúng:", Solution().minAbsoluteSumDiff(a, b), "| vét cạn:", min_sum_brute(a, b))

tham lam: 7 | đúng: 6 | vét cạn: 6


**Vì sao sai:** vị trí 0 chênh lớn nhất (`|8 − 4| = 4`) nhưng không có giá trị nào trong `{8, 9}` gần 4 hơn ⇒ lợi ích 0. Vị trí 1 chênh 3 nhưng thay `9 → 8` giảm còn 2 ⇒ lợi ích 1. Phải tính **lợi ích** từng vị trí, không phải độ chênh. (Phản ví dụ tự dựng, đối chiếu vét cạn.)

### Lỗi 3 — Lấy modulo rồi mới so sánh / chọn min

In [22]:
def wrong_mod_compare(nums1, nums2):
    srt = sorted(nums1); n = len(srt)
    S = sum(abs(x - y) for x, y in zip(nums1, nums2))
    best_total = S % MOD                                  # SAI: so sánh trên giá trị đã lấy dư
    for a, b in zip(nums1, nums2):
        p = bisect_left(srt, b)
        d2 = min(abs(srt[q] - b) for q in (p - 1, p) if 0 <= q < n)
        best_total = min(best_total, (S - abs(a - b) + d2) % MOD)
    return best_total

print("sai:", wrong_mod_compare(big1, big2), "| đúng:", Solution().minAbsoluteSumDiff(big1, big2))

sai: 89992 | đúng: 999990000


**Vì sao sai:** tổng ban đầu `1000089999` lấy dư thành `89992` — "nhỏ" một cách giả tạo, nên thuật toán tưởng không thay là tốt nhất. Thứ tự giữa các số **không** được giữ sau khi lấy dư (mục 3.4). **Sửa:** tìm min/max trên giá trị thật rồi lấy dư một lần.

### Lỗi 4 — Thay bằng giá trị bất kỳ / lấy từ `nums2`

In [23]:
def wrong_any_value(nums1, nums2):
    S = sum(abs(x - y) for x, y in zip(nums1, nums2))
    return (S - max(abs(x - y) for x, y in zip(nums1, nums2))) % MOD   # SAI: coi như thay a_i = b_i

for a, b, want in examples:
    print("sai:", wrong_any_value(a, b), "| đúng:", want)

sai: 1 | đúng: 3
sai: 0 | đúng: 0
sai: 19 | đúng: 20


**Vì sao sai:** đề chỉ cho thay bằng **một phần tử của `nums1`**, không phải giá trị tùy ý (đặt `a_i = b_i` sẽ xóa hẳn chênh lệch). Với Ví dụ 3 cách sai cho `27 − 8 = 19 < 20` — đẹp nhưng không hợp lệ.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- **"Được làm tối đa một thao tác để tối ưu tổng"** ⇒ tổng ban đầu ± **lợi ích tốt nhất** của một thao tác; lợi ích thường tính độc lập theo vị trí.
- **"Phần tử gần `x` nhất trong một tập cố định"** ⇒ sắp xếp + `bisect_left`, xét `p` và `p − 1`.
- Nhiều truy vấn và có thể sắp xếp truy vấn ⇒ hai con trỏ thay cho nhị phân.
- Modulo chỉ ở kết quả; **không** so sánh giá trị đã lấy dư.
- Nghi ngờ tham lam "sửa chỗ tệ nhất" — kiểm bằng vét cạn.

## 15. Biến thể của bài

1. **Thay tối đa `k` phần tử** (giá trị lấy từ `nums1` ban đầu): lợi ích các vị trí độc lập ⇒ lấy `k` lợi ích lớn nhất (Bài tập 5).
2. **Thay bằng giá trị bất kỳ trong một tập `T` khác** ⇒ sắp xếp `T` thay vì `nums1`.
3. **Tổng bình phương chênh lệch** `Σ (a_i − b_i)²` ⇒ cùng ý tưởng, đổi hàm khoảng cách (gần nhất vẫn là `p` hoặc `p − 1`).
4. **Hoán đổi hai phần tử của `nums1`** (thay vì thay thế) ⇒ khó hơn: lợi ích phụ thuộc cặp vị trí.
5. Bài cùng kỹ thuật "gần nhất bằng bisect": LeetCode 2033, 1300 *Sum of Mutated Array Closest to Target*, 1847 *Closest Room*.

## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Thử mọi phép thay | tính lại tổng | `O(n³)` | `O(n)` | chuẩn đối chiếu |
| 2. Lợi ích + quét | `gain_i`, gần nhất bằng quét | `O(n²)` | `O(1)` | `n` ≤ vài nghìn |
| 3. Sort + bisect | gần nhất bằng nhị phân | `O(n log n)` | `O(n)` | **nộp bài** |
| 4. Sort + hai con trỏ | truy vấn theo thứ tự | `O(n log n)` | `O(n)` | nhiều truy vấn, tránh `log` lặp |

## 17. 5 bài tập từ dễ đến khó

**Bài 1 (đoán output):** `([1, 2, 3], [3, 2, 1])` và `([5], [1])`?

**Bài 2 (tính tay):** Lập bảng lợi ích cho `nums1 = [8, 9]`, `nums2 = [4, 6]`.

**Bài 3 (điền TODO):** Viết `closest_value(srt, x)` trả về giá trị trong mảng tăng `srt` gần `x` nhất (hòa thì lấy giá trị nhỏ hơn).

**Bài 4 (tự viết lại):** Viết `best_replacement(nums1, nums2)` trả về `(tổng_mới_thật, i, giá_trị_thay)` — một phép thay tối ưu (hoặc `(S, -1, None)` nếu không cần thay).

**Bài 5 (mini challenge):** Cho phép thay **tối đa `k`** phần tử (mỗi giá trị thay lấy từ `nums1` **ban đầu**). Viết `min_sum_k(nums1, nums2, k)` trả về tổng nhỏ nhất (không modulo).

In [24]:
def closest_value(srt, x):
    # TODO
    pass

def best_replacement(nums1, nums2):
    # TODO
    pass

def min_sum_k(nums1, nums2, k):
    # TODO
    pass

## 18. Gợi ý 3 mức cho bài tập

| Bài | Gợi ý 1 | Gợi ý 2 | Gợi ý 3 |
|---|---|---|---|
| 1 | Tổng ban đầu `2 + 0 + 2 = 4` | Thay `1 → 3` ở vị trí 0 | Đáp án 2; `[5],[1]` → 4 |
| 2 | `i=0`: gần 4 nhất là 8 | `i=1`: gần 6 nhất là 8 | Lợi ích `[0, 1]` |
| 3 | `p = bisect_left(srt, x)` | Ứng viên `srt[p-1]`, `srt[p]` | So `x - srt[p-1] <= srt[p] - x` |
| 4 | Lưu thêm chỉ số đạt lợi ích lớn nhất | Giá trị thay = `closest_value(srt, b_i)` | `S − gain` |
| 5 | Lợi ích các vị trí **độc lập** | Sắp xếp lợi ích giảm dần | Trừ `k` lợi ích lớn nhất |

## 19. Đáp án tham khảo

In [25]:
r1 = [Solution().minAbsoluteSumDiff(a, b) for a, b in [([1, 2, 3], [3, 2, 1]), ([5], [1])]]
print("Bài 1:", r1); assert r1 == [2, 4]
print("Bài 2:"); assert trace_bisect([8, 9], [4, 6]) == 6

Bài 1: [2, 4]
Bài 2:
sorted(nums1) = [8, 9]
i=0 a=8 b=4 d1=4 p=0 trái=None phải=8 d2=4 lợi ích=0 S=4 max=0
i=1 a=9 b=6 d1=3 p=0 trái=None phải=8 d2=2 lợi ích=1 S=7 max=1
đáp án = 6


In [26]:
def closest_value(srt, x):
    p = bisect_left(srt, x)
    if p == 0: return srt[0]
    if p == len(srt): return srt[-1]
    return srt[p - 1] if x - srt[p - 1] <= srt[p] - x else srt[p]

def best_replacement(nums1, nums2):
    srt = sorted(nums1)
    S = sum(abs(x - y) for x, y in zip(nums1, nums2))
    best = (0, -1, None)
    for i, (a, b) in enumerate(zip(nums1, nums2)):
        v = closest_value(srt, b)
        g = abs(a - b) - abs(v - b)
        if g > best[0]:
            best = (g, i, v)
    return S - best[0], best[1], best[2]

rng = random.Random(4)
for _ in range(500):
    srt = sorted(rng.randint(1, 30) for _ in range(rng.randint(1, 8)))
    x = rng.randint(0, 32)
    assert closest_value(srt, x) == min(srt, key=lambda v: (abs(v - x), v))
    a = [rng.randint(1, 12) for _ in range(rng.randint(1, 7))]; b = [rng.randint(1, 12) for _ in range(len(a))]
    tot, i, v = best_replacement(a, b)
    assert tot == min_sum_brute(a, b)
    if i != -1:
        arr = a[:]; arr[i] = v
        assert sum(abs(x - y) for x, y in zip(arr, b)) == tot and v in a
print("Bài 3, 4: OK", best_replacement([1, 10, 4, 4, 2, 7], [9, 3, 5, 1, 7, 4]))

Bài 3, 4: OK (20, 0, 10)


In [27]:
from itertools import combinations, product

def min_sum_k(nums1, nums2, k):
    srt = sorted(nums1)
    S, gains = 0, []
    for a, b in zip(nums1, nums2):
        d1 = abs(a - b); S += d1
        gains.append(d1 - abs(closest_value(srt, b) - b))
    gains.sort(reverse=True)
    return S - sum(gains[:k])

def min_sum_k_brute(nums1, nums2, k):
    n = len(nums1); vals = sorted(set(nums1)); best = None
    for m in range(0, min(k, n) + 1):
        for idx in combinations(range(n), m):
            for choice in product(vals, repeat=m):
                arr = nums1[:]
                for i, v in zip(idx, choice):
                    arr[i] = v
                tot = sum(abs(x - y) for x, y in zip(arr, nums2))
                best = tot if best is None else min(best, tot)
    return best

rng = random.Random(6)
for _ in range(300):
    n = rng.randint(1, 5)
    a = [rng.randint(1, 9) for _ in range(n)]; b = [rng.randint(1, 9) for _ in range(n)]
    k = rng.randint(0, 3)
    assert min_sum_k(a, b, k) == min_sum_k_brute(a, b, k), (a, b, k)
    assert min_sum_k(a, b, 1) == Solution().minAbsoluteSumDiff(a, b)
print("Bài 5 (biến thể tự dựng — đối chiếu lại đề gốc nếu có): OK", min_sum_k([1, 10, 4, 4, 2, 7], [9, 3, 5, 1, 7, 4], 2))

Bài 5 (biến thể tự dựng — đối chiếu lại đề gốc nếu có): OK 14


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
srt = sorted(nums1); n = len(srt)
total = best_gain = 0
for a, b in zip(nums1, nums2):
    d1 = abs(a - b); total += d1
    p = bisect_left(srt, b)
    d2 = d1
    if p < n: d2 = min(d2, srt[p] - b)
    if p > 0: d2 = min(d2, b - srt[p - 1])
    best_gain = max(best_gain, d1 - d2)
return (total - best_gain) % MOD
```

**Checklist:**
- [ ] Tìm gần nhất trong **`nums1` ban đầu** (đã sắp xếp), không phải `nums2`?
- [ ] Xét **cả** `srt[p]` và `srt[p − 1]`, kiểm biên `p`?
- [ ] Tối đa hóa **lợi ích**, không phải sửa chỗ chênh lớn nhất?
- [ ] Modulo **sau** khi trừ, không so sánh giá trị đã lấy dư?
- [ ] `n = 1` chạy đúng (lợi ích 0).